In [1]:
# Loading libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# Loading the Medicare 2024 dataset and preprocessing it
cols = ["Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc", "Rndrng_Prvdr_Cntry",
        "Avg_Mdcr_Stdzd_Amt", "Avg_Mdcr_Alowd_Amt"]
df = pd.read_parquet("medicare_2024.parquet", columns=cols)

df = df[(df["Rndrng_Prvdr_Cntry"] == "US") & (df["Avg_Mdcr_Alowd_Amt"] > 0)]
df["log_pay"] = np.log(df["Avg_Mdcr_Stdzd_Amt"].clip(lower=1e-6))
len(df)

9781233

In [2]:
def eta_sq(group_cols):
    group_mean = df.groupby(group_cols, observed=True)["log_pay"].transform("mean")
    total = ((df["log_pay"] - df["log_pay"].mean()) ** 2).sum()
    within = ((df["log_pay"] - group_mean) ** 2).sum()
    return 1 - within / total

groupings = {
    "Type": ["Rndrng_Prvdr_Type"],
    "Type x POS": ["Rndrng_Prvdr_Type", "Place_Of_Srvc"],
    "HCPCS": ["HCPCS_Cd"],
    "HCPCS x POS": ["HCPCS_Cd", "Place_Of_Srvc"],
    "Type x HCPCS x POS": ["Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc"],
}
eta = pd.Series({name: eta_sq(c) for name, c in groupings.items()}).sort_values()
eta.round(3)

Type                  0.131
Type x POS            0.181
HCPCS                 0.928
HCPCS x POS           0.957
Type x HCPCS x POS    0.975
dtype: float64

In [3]:
rng = np.random.default_rng(42)

t1 = ["Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc"]
df["group"] = df.groupby(t1, observed=True).ngroup()
sizes = df["group"].value_counts()

# 200 random big groups (300+ providers) as "ground truth"
big = rng.choice(sizes[sizes >= 300].index, size=200, replace=False)
groups = df[df["group"].isin(big)].groupby("group")["log_pay"]

rows = []
for g, x in groups:
    x = x.to_numpy()
    mad_full = np.median(np.abs(x - np.median(x)))
    if mad_full == 0:
        continue
    for n in [5, 10, 15, 20, 30, 50]:
        s = rng.choice(x, size=(100, n))   # 100 samples of n providers
        mad_n = np.median(np.abs(s - np.median(s, axis=1, keepdims=True)), axis=1)
        rows.append((n, np.median(np.abs(mad_n / mad_full - 1))))

stab = pd.DataFrame(rows, columns=["n", "mad_error"]).groupby("n")["mad_error"].median()
stab.round(3)

n
5     0.492
10    0.323
15    0.271
20    0.228
30    0.182
50    0.142
Name: mad_error, dtype: float64

In [4]:
n_rows = df["group"].map(sizes)
for k in [10, 20, 30, 50]:
    print(f"min {k}: {(n_rows < k).mean():.2%} of rows need a fallback group")

min 10: 1.04% of rows need a fallback group
min 20: 1.77% of rows need a fallback group
min 30: 2.37% of rows need a fallback group
min 50: 3.37% of rows need a fallback group


Minimum peer group size. With 10 providers per group, the MAD(median absolute deviation) is typically off by ~32%, so a row with a true robust z of 3 would typically get a z somewhere between ~2.3 and ~4.4 (3 / 1.32 and 3 / 0.68). With 30 providers the error drops to ~18%.

Raising the threshold from 10 to 30 moves only 1.3% more rows to a fallback group (1.04% to 2.37%), while 97.6% of rows keep the most specific peer group.

Therefore I suggest MIN_PEERS = 30. Since more rows now use fallback groups, the fallback should keep HCPCS (HCPCS x POS), not Type x POS.

In [5]:
cols = ["Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc", "Rndrng_Prvdr_Cntry",
        "Tot_Benes", "Tot_Srvcs", "Tot_Bene_Day_Srvcs",
        "Avg_Sbmtd_Chrg", "Avg_Mdcr_Alowd_Amt", "Avg_Mdcr_Stdzd_Amt"]
df = pd.read_parquet("medicare_2024.parquet", columns=cols)

# same cleaning as data_preprocessing.ipynb
df = df[(df["Rndrng_Prvdr_Cntry"] == "US")
        & (df["Avg_Mdcr_Alowd_Amt"] > 0)
        & (df["Tot_Srvcs"] >= df["Tot_Benes"])
        & (df["Tot_Bene_Day_Srvcs"] <= df["Tot_Srvcs"])]

# 3 features: how much Medicare pays, how intensive, how much is overcharged
df["log_pay"] = np.log(df["Avg_Mdcr_Stdzd_Amt"].clip(lower=1e-6))
df["srvcs_per_bene"] = np.log(df["Tot_Srvcs"] / df["Tot_Benes"])
df["markup"] = np.log(df["Avg_Sbmtd_Chrg"].clip(lower=1e-6) / df["Avg_Mdcr_Alowd_Amt"])
len(df)

9781161

In [6]:
MIN = 30
levels = [["Rndrng_Prvdr_Type", "HCPCS_Cd", "Place_Of_Srvc"],   # level 1
          ["HCPCS_Cd", "Place_Of_Srvc"],                        # level 2 (fallback)
          ["HCPCS_Cd"]]                                         # level 3 (fallback)

df["level"] = 0
df["peer"] = -1
for i, c in enumerate(levels, start=1):
    size = df.groupby(c, observed=True)["log_pay"].transform("size")
    gid = df.groupby(c, observed=True).ngroup() + i * 10_000_000   # unique id per level
    take = (df["level"] == 0) & (size >= MIN)
    df.loc[take, "peer"] = gid[take]
    df.loc[take, "level"] = i

df["level"].value_counts(normalize=True).mul(100).round(2)   # 0 = unscored

level
1    97.63
2     2.03
0     0.24
3     0.10
Name: proportion, dtype: float64

In [11]:
feats = ["log_pay", "srvcs_per_bene", "markup"]
s = df[df["level"] > 0]   # only rows that have a peer group

med = s.groupby("peer")[feats].transform("median") # median of each feature within the peer group
dev = (s[feats] - med).abs()   # absolute distance of each row from its group median
mad = dev.groupby(s["peer"]).transform("median") # MAD = typical distance from the median (robust spread)

print("rows where MAD = 0 (%):")  # MAD = 0  then more than half of the group has the same value
print((mad == 0).mean().mul(100).round(1))

# scale ~ standard deviation: 1.4826 * MAD; if MAD = 0, fall back to 1.2533 * mean absolute deviation
scale = (1.4826 * mad).where(mad > 0, 1.2533 * dev.groupby(s["peer"]).transform("mean"))
rz = (s[feats] - med) / scale.where(scale > 0)            # robust z

rows where MAD = 0 (%):
log_pay           10.9
srvcs_per_bene    31.6
markup             0.3
dtype: float64


In [12]:
flags = rz.abs() > 3.5  # True if the feature is unusual (|robust z| > 3.5)
print("% of rows flagged per feature:")
print(flags.mean().mul(100).round(2)) # share of rows flagged for each feature

n_flags = flags.sum(axis=1)   # number of unusual features per row (0-3)
print(f"\nunusual on at least one feature: {(n_flags > 0).mean():.2%}")
print(f"unusual on 2+ features:          {(n_flags >= 2).mean():.2%}")

% of rows flagged per feature:
log_pay           6.44
srvcs_per_bene    3.69
markup            1.64
dtype: float64

unusual on at least one feature: 11.24%
unusual on 2+ features:          0.51%


In [13]:
for f in feats:
    m0 = mad[f] == 0
    print(f"{f:15s} flagged when MAD=0: {flags.loc[m0, f].mean():6.1%}   when MAD>0: {flags.loc[~m0, f].mean():6.1%}")

log_pay         flagged when MAD=0:   1.5%   when MAD>0:   7.0%
srvcs_per_bene  flagged when MAD=0:   4.6%   when MAD>0:   3.3%
markup          flagged when MAD=0:   5.7%   when MAD>0:   1.6%


In [14]:
spread = (1.4826 * mad["log_pay"])
print(f"typical within-group spread of payment: {np.expm1(spread.median()):.1%}")
print(f"so |z| > 3.5 means paid more than ~{np.expm1(3.5 * spread.median()):.0%} above/below the group median")

typical within-group spread of payment: 5.6%
so |z| > 3.5 means paid more than ~21% above/below the group median


The rule-based baseline (robust z within peer groups) will serve as a transparent benchmark: we will check how much its flags overlap with Isolation Forest's top-ranked rows.